In [ ]:
import pyodbc
import pandas as pd

In [ ]:
# To check what Microsoft Access Drivers exist
[x for x in pyodbc.drivers() if x.startswith('Microsoft Access Driver')]

# Graduation Rate Database

Given our research questions, the columns we need are:
- School year
- LEA_BEDS (12 digit code to specify category/subcategory of data sample)
- GRAD_PCT (denotes percentage of students who graduated in that LEA_BED)

In [ ]:
DB_FILE = r'..\data\raw\gradrate\2025 High School Graduation Rate Researcher File.mdb' 
CONN_STR = (
    r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
    rf"DBQ={DB_FILE};"
)

In [ ]:
conn = pyodbc.connect(CONN_STR)

In [ ]:
df = pd.read_sql('SELECT * FROM GRAD_RATE_AND_OUTCOMES_2025', conn)
df

In [ ]:
og = df

In [ ]:
df.columns

- Whenever last 4 digits are not '0000' in lea_beds, it means that it is a charter school


- ENTITY_CD is same as AGGREGATION_CODE in the two datasets

In [ ]:
df.loc[9360]

In [ ]:
# Get all rows aggregated by district (aggregation_index == 3), and get all columns needed
df = df.loc[df.AGGREGATION_INDEX == '3', ['REPORT_SCHOOL_YEAR', 'AGGREGATION_TYPE', 'AGGREGATION_CODE', 'AGGREGATION_NAME','MEMBERSHIP_CODE',
       'MEMBERSHIP_DESC', 'subgroup_code', 'SUBGROUP_NAME', 'ENROLL_CNT',
       'GRAD_CNT', 'GRAD_PCT']]
df

In [ ]:
df.to_csv("../data/processed/grad_pct_per_DISTRICT.csv", index=False)

In [ ]:
conn.close()

# Enrollment database

Columns we need:

- ENTITY_CD (12 digit code to specify category/subcategory of data sample)
- ENTITY_NAME
- YEAR
- PER_ECDIS

In [ ]:
DB_FILE = r'..\data\raw\ENROLLMENT_2025\ENROLL2025_20251217.mdb' 
CONN_STR = (
    r"DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};"
    rf"DBQ={DB_FILE};"
)

conn = pyodbc.connect(CONN_STR)

In [ ]:
df = pd.read_sql('SELECT ENTITY_CD, ENTITY_NAME, YEAR, PER_ECDIS FROM "Demographic Factors"', conn)
df

In [ ]:
# Keep only rows which are (most likely) districts
df = df.loc[(df.ENTITY_CD.str[-4:] == '0000') & (df.ENTITY_CD.str[:2] != '00')]
df

In [ ]:
df.to_csv('../data/processed/PER_ECDIS_per_DISTRICT.csv', index=False)

In [ ]:
conn.close()